<!--- Made by:
      Oscar Antonio Restrepo Gutiérrez
--->

# Teoría de errores en computación

Es importante entender que los computadores no calculan de manera exacta sino aproximada en la mayoría de los casos. Una solución exacta se obtiene solo cuando se trabaja con números enteros, si las operaciones son con números de punto flotante el desarrollo del algoritmo tendrá aproximaciones debido al truncamiento en la operaciones (esto a causa de la representación de números de punto flotante de 32 o 64 bits). Una forma de reducir el error es aumentar el número de bits (por ejemplo a 128 bits) es decir, a más bits más precisión pero nunca una solución igual a la real obtenida por el método analítico. Como veremos las operaciones algebraicas de suma y resta no son exactas ($(a + b) + c \neq a + (b + c)$ es decir, no es asociativa) debido a la representación flotante.

## *Overflow* y *underflow* 
Se refiere al desbordamiento aritmético que se da cuando el binario almacenado en un registro supera su valor máximo, es decir se requiere más bits de los permitidos, este se da con números muy grandes o muy pequeños,  
```c
      Overflow }---------------{----0----}--------------{ Overflow
                                underflow
```
El rango de un doble (64 bits) está en el intervalo $10^{-322} < x < 10^{308}$, ósea que
cualquier número fuera de este rango necesita más de 64 bits para ser representado, si $x$ es muy grande hay *overflow* y si es muy pequeño hay *underflow*. La siguiente rutina nos permite calcular el *overflow* multiplicando por 2 y el *underflow* dividiendo por 2,
```python    
i=0.0 
under = over = 1.0
while i<1100:
    under = under/2.0
    over = over*2.0
    print (i, under, over)
    i = i+1
```
Si hay *overflow* python imprime *inf* que significa infinito y si hay underflow python imprime 0.0. Note que una vez alcanzado el *underflow* `under` es más pequeño que el *épsilon de la máquina*. 

## Épsilon de la máquina
<a id='epsilon_maquina'></a>
Se refiere al número más pequeño o grande que se puede adicionar en una suma sin que esta cambie, es decir $x+\epsilon_{min}=x$ (o $x+\epsilon_{max}=\epsilon_{max}$). En otras palabras, *"los números que difieren por menos que el épsilon de la máquina, $\epsilon$, son numéricamente iguales"*, esto se da debido al truncamiento de la máquina por usar números flotantes.
 
En python se puede calcular la precisión numérica de la máquina con la siguiente rutina
```python      
# Épsilon de la máquina: el eps más pequeño tal que 1 + eps != 1
eps = 1 # epsilon inicial
for n in range(1200):
    eps = eps/2.0
    one = 1. + eps
    print(n, one, eps)
```
Note que en cada paso `one` es la suma de `1 + eps`, donde `eps` se reduce a la mitad en cada iteración.
¿Cuántos iteraciones se necesitan para que `one` sea igual a `1.0 + eps`, es decir que `eps` sea tan pequeño que no aporte a la suma?

Python tiene otras formas adicionales de calcular el `eps` con mayor precisión usando numpy: 
```python 
import numpy as np
eps = np.finfo(float).eps 
1.0 == 1.0 + eps/2. # resultado: True
```    
(Verificar en ipython estas dos lineas de comandos para verificar que 1 es igual a 1 + `eps/2`; compare el valor de `eps` al valor encontrado en el caso anterior al rededor de la iteración 51 ¿son iguales?).

Igualmente se puede encontrar, para números grandes, el punto en el que sumarles 1.0 deja de tener efecto:
```python      
# Precisión en la suma cuando el segundo sumando (eps) es mucho más grande
eps=1
for n in range(1200):
    eps=eps*2.0
    one=1.0 + eps
    print(n, one, eps, one-eps) 
```
Note que a cada paso se adiciona un `eps` multiplicado por 2. Igual que en el caso anterior hay un momento en el cual `one - eps` se hace cero (¿a qué paso?), ósea: `eps` es tan grande que la la adición de 1.0 no cambia el resultado debido a que esta adición queda fuera del rango de representación del número (para ver esto corra el código y compare `one` y `eps`, ¿a qué paso ocurre esto?).

In [ ]:
i=0.0 
under = over = 1.0
while i<1100:
    under = under/2.0
    over = over*2.0
    print (i, under, over)
    i = i+1

In [ ]:
# Épsilon de la máquina: el eps más pequeño tal que 1 + eps != 1
eps = 1 # epsilon inicial
for n in range(1200):
    eps = eps/2.0
    one = 1. + eps
    print(n, one, eps)


In [ ]:
# Precisión en la suma cuando el segundo sumando (eps) es mucho más grande
eps = 1 # epsilon inicial
for n in range(1200):
    eps = eps*2.0
    one = 1. + eps
    print(n, one, eps)

## Tipos de errores
<a id='Tipos_de_errores'></a>
Primero que todo miremos los tipos de errores que se dan en computación: 

**Errores del usuario**:  son errores tipográficos, mal diseño del código, archivo equivocado, etc, estos son evitables.

**Errores aleatorios**: Fluctuaciones eléctricas, fallo de electricidad, rayos cósmicos etc, estos errores son raros y la probabilidad de que ocurran aumentan con el tiempo de computo del algoritmo en la maquina (hay códigos que corren por semanas o meses). En estos errores no se tiene control como en el caso anterior. 

**Errores de aproximación**: Estos errores son más de carácter matemático, es decir aproximaciones que se hacen al truncar una serie o la solución de una ecuación, la serie de Taylor es la herramienta preferida para este tipo de aproximaciones:

$$\sin(x) = \sum^{N}_{n=1} \frac{(-1)^{n-1}}{(2n-1)!} x^{2n-1} + \epsilon(x,N),$$
donde $ \epsilon(x,N)$ es el error introducido al truncar la serie.

**Errores de redondeo numérico**: Ocurre por solo considerar un número finito de cifras significativas en las operaciones numéricas. Por Ejemplo, los números fraccionarios 2/3 y 1/3 tienen infinitas cifras en base 10 (y en base 2), al truncar con 4 cifras significativas tenemos $1/3\approx0.3333$, pero $2/3\approx0.6667$, pues la ultima cifra se redondea a 7, así si hacemos la operación: 
$$2\times\frac{1}{3}-\frac{2}{3}=0.6666-0.6667=-\,0.0001\neq0$$
encontramos un error de 0.0001, aunque este valor es pequeño no es cero.

**Errores debido al redondeo por uso de números de tipo flotante**: Este error es básicamente del mismo tipo del caso anterior, se debe a que al usar una representación finita de 32 o 64 bits se hace una truncación numérica de 7 cifras significativas para 32 bits y de aproximadamente 15 o 16 cifras para 64 bits (recordar la representación binaria). Afortunadamente los computadores modernos ya usan representación de 64 bits que permite reducir considerablemente este tipo de error. No obstante miremos un poco más en detalle este tipo de error y como afecta las aproximaciones matemáticas:

In [ ]:
import numpy as np

# Error en operaciones simples para 16, 32 y 64 bits:
print( np.float16(5/7.),np.float32(5/7.), 5/7)
print( np.float16(1/10.),np.float32(1/10.), 1/10)

In [ ]:
# Error en la suma con 16 bits de representación, 
# comparar a resultado con 64 bits (usado por defecto)
# note las funciones numpy.floatxx redondean a 16,32,64 o 128 bits

# sumar 1/10 diez veces no da 1.0
x = 0
for i in range(10):
    x += np.float16(1.0/10)
print('operación con 16 bits:',x)

**Preguntas**:<br>
Si un número en representación de 16 bits usa 5 bits para el exponente y 10 para la mantisa ¿Cuántas cifras significativas tiene en base 10?
¿Cuáles cifras son basura en el cálculo anterior?

**Ejemplo**, la siguiente multiplicatoria, $\prod_{n=1}^{20}2^{\frac{1}{20}}$,
debe converger a 2, comparemos los resultados a 16 y 64 bits:

In [ ]:
# Error en multiplicación con 16 bits de representación, 
# comparar a resultado con 64 bits (usado por defecto) 
#(resta y division son casos especiales de adición y multiplicación.)
# la multiplicatoria converge a 2.0
N = 20
x16 = x64 = 1
print ('iter 16 bits,   64 bits,     error')
for i in range(N):
    x16 *= np.float16(2.0**(1.0/N))
    x64 *= np.float64(2.0**(1.0/N))#Note que no es necesario escribir el float64
    
print (i, x16, x64, np.abs(x16-x64))

El siguiente ejemplo muestra el error en el cálculo de la serie de la función seno, 

$$\sin(x) = \sum^{N}_{n=1} \frac{(-1)^{n-1}}{(2n-1)!} x^{2n-1} + \epsilon(x,N),$$

debido al uso de 32 bits en vez de 64 bits. 
Este algoritmo es pesado debido al factorial, el error con 32 bits es obvio pero no con 64 bits.

Importante: Note que cada número tiene que ser pasado a 32 bits para que la operación numérica sea de 32 bits, si algún número no es de 32 bits python pasará automáticamente todos los números a 64 bits en las operaciones, y no se podrá ver fácilmente el error.

In [ ]:
#------ error en el calculo de la serie seno con 80 terms----------------
# probar con otros ángulos!
import numpy as np
import math

x = np.float32(85*np.pi/180) # Inicialize   
N = 80                   # usar un valor mas grande da error
f_1 = np.float32(1); f_2 = np.float32(2)
Sum = np.float32(0.0)    # observe que se pone 0.0 (flotante) y no 0 (entero),
for n in range(1,N+1):   # da el array 1,2,3, ..... N
   nf = np.float32(n)
   Sum = Sum + (-f_1)**(nf-f_1)*x**(f_2*nf-f_1)/np.float32(math.factorial(2*n-1))

print('sen(x),         serie')    
print(np.sin(85*np.pi/180), Sum) # use Sum.dtype para verificar que es un float32

In [ ]:
import math
# Este algoritmo es igual al anterior, pero por defecto con 64 bits
# y se usa la función numpy.sum()
x = 85*np.pi/180 # Inicialize   
N = 80                     # usar un valor mas grande da error

Sum = sum([(-1.)**(n-1.)*x**(2.*n-1.)/math.factorial(2*n-1) for n in range(1,N+1)])
print (Sum, abs(Sum-np.sin(85*np.pi/180))) 

## Cómo medir errores
Sea $x$ el valor verdadero y $x^*$ el valor aproximado

**Error absoluto**: se define como 
\begin{equation*} 
\epsilon_{abs}= |x-x^*|
\end{equation*}
**Error relativo**: es dado por 
\begin{equation*} 
\epsilon_{rel}= \frac{|x-x^*|}{|x|}
\end{equation*}
**Error porcentual**: es dado por 
\begin{equation*} 
\epsilon_{\%}= \frac{|x-x^*|}{|x|}*100
\end{equation*}
**Error en series**: El error para truncar una serie se toma como
\begin{equation*} 
\epsilon_{aprox}= \left|\frac{nth\hbox{-term}}{\hbox{suma}}\right|< \hbox{eps}
\end{equation*}
La tolerancia normalmente se toma como un número pequeño, por ejemplo `eps` $=10^{-10}$. Note que no se trunca la serie usando $|{nth}\hbox{-term}|<$ eps,   usar esta forma puede conducir a errores debido a que no se compara con el valor de la suma (un millón comparado a uno es grande, pero comparado a diez mil millones es pequeño).

Tomemos como ejemplo otra vez el cálculo de la serie del seno y calculemos el error,

In [ ]:
import math
# Solo de 8 a 9 pasos son necesitados para alcanzar la precision epsilon (eps).
x = 85*np.pi/180;  eps = 1.0e-8 # Inicialize   
N = 80      # usar un valor mas grande da error
Sum = 0.0   # observe que se pone 0.0 (flotante) y no 0 (entero),
for n in range(1,N+1): # da el array 1,2,3, ..... N
   term = (-1)**(n-1)*x**(2.*n-1.)/math.factorial(2*n-1)
   Sum = Sum + term
   if ( abs (term/Sum) < eps ): break # parar si el error es menor que epsilon
   
print (Sum, n, abs (term/Sum)) 

El calculo del factorial tiene varios detalles importantes, pues crece de manera rápida (18! ya tiene 16 digitos) y aunque python3 tiene una precisión arbitraria para enteros, en algunos casos se debe multiplicar por un float y el resultado finál quedará truncado a unos 16 dígitos (los restantes se pierden) veamos:

In [ ]:
import math

# Problema del factorial al multiplicar por un float:
math.factorial(60), math.factorial(60)*1.0

Note que al multiplicar por 1.0 el punto se corre de la posición 82 a la 1 y se trunca a 15 dígitos, esto introduce errores en los cálculos. Además el factorial de un número no muy grande produce desbordamiento númerico (por ejemplo $171!$ tiene 310 dígitos y $171!\times 1.0$ da un error pues es del orden de $10^{310}$ que es más grande que el mayor exponente permitido para floats). 

### Reciclaje de cálculos y reducción del error
<a id='reciclaje_de_variable'></a>
Finalmente resaltemos la importancia de reciclar los términos calculados en iteraciones previas. Primero que todo note que el error en el cálculo de la serie seno se da debido a que para el *nth*-término se calcula la división de dos cantidades muy grandes, esto introduce un error significativo, en la siguiente variación se evita calcular el factorial, el resultado es mejor que en los dos ejemplos anteriores, note que se reescribe el término de la serie como, 

$$\frac{(-1)^{n-1}}{(2n-1)!} x^{2n-1}=\frac{-x^2}{(2n-1)(2n-2)} \frac{(-1)^{n-2}}{(2n-3)!} x^{2n-3}$$
(Tarea, verificar esta igualdad).

In [ ]:
# Funciona para x distinto de cero.
Sum = term = x = 85*np.pi/180; eps = 1.0e-8 # Inicialize
n = 2 # comenzamos en 2, pues para n = 1 ya asignamos el valor x
while ( abs (term/Sum) > eps ):
   term = -term*x*x/(2.*n-1.)/(2.*n-2.)
   Sum = Sum + term
   n = n + 1

print('función seno:',np.sin(x))
print('serie seno:  ',Sum, n, abs (term/Sum)) 

En general la forma más eficiente de calcular la serie del seno es reciclar
el resultado del paso anterior, evitando introducir el error producido por un  overflow. 

**Problema**: El error por no reciclar es bastante notable con 32 bits, con 64 bits casi no se nota; repita este problema usando 32 bits y compare.

**Problema**: el código anterior de la serie seno falla para $x=0$ ¿cómo lo solucionaría? 


### Error al sumar  cantidades grandes con pequeñas
Cuando se suman cantidades pequeñas a cantidades grandes, las dos cantidades no deben diferir por una cantidad que  requiera mayor cantidad de cifras significativas que la representación, por ejemplo la siguiente suma con con 64 bits da el mismo resultado,
```python 
      1.0 + 1e16 = 1e16, 
```      
como se explicó [anteriormente](#epsilon_maquina), esto se da debido a que usamos 64 bits y esta representación solo permite unas 15 cifras significativas, note que si sumamos, 
```python       
      1.0 + 1.0e15 = 1000000000000001.0
```
el uno se agrega en la último digito significativo (en este caso el 15).
Esto significa que la suma no es conmutativa en números de punto flotante, es decir, $(a + b) + c \neq a + (b + c)$.      

In [ ]:
# Depende de como se asocien los unos sumados da resultados diferentes: 
print(1e16 + 1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1  )
print(1e16 +(1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1) )
print(1e16 +(1+1+1+1+1+1+1+1+1+1+1+1+1)+1+1+1+1+1+1+1+1+1+1+1 )

### Error de cancelación sustractiva
<a id='Error_de_cancelación_sustractiva'></a>
Ocurre cuando se restan dos números muy similares, debido a que el resultado cuenta con menos cifras significativas, por ejemplo considere la siguiente operación con 64 bits,
```python 
       1234567891 - 1234567809 = 82
```      
pero con 32 bits da,
```python 
      float32(1234567891) - float32(1234567809) = 128.0
```
Si esta resta está en el denominador de una operación el error es todavía peor (calcule los inversos y compare).

La precisión en el resultado de la cancelación será igual al número de cifras a la derecha que sean distintas de cero, por ejemplo la resta, $1234999-1234888=0000111$, tiene solo 3 cifras significativas. 

**Ejemplo**

In [ ]:
# Operación con 64 bits
x = 1234567891.0 
y = 1234567809.0 
print(x - y)   # da 82.0

# Operación con 32 bits
x32 = np.float32(x)
y32 = np.float32(y)
print(x32 - y32) # da 128.0 

**Ejercicio**: comparar las siguientes dos cantidades donde los primeros 8 dígitos son iguales, restar para ver que la precisión del resultado es de ocho dígitos (los últimos 8 dígitos) y no de 16:

In [ ]:
# Compare las siguientes cantidades para tipo int y tipo float
# primeros 8 digitos iguales, la resta da: 00000000806398288.0
12345678901234567, 12345678094836279,  12345678901234567., 12345678094836279. 

**Ejemplo**: comparación de dos cantidades con 17 dígitos iguales:

In [ ]:
# primeros 17 digitos iguales con enteros: resultado verdadero
123456789012345675454325 - \
123456789012345678794305 # 

In [ ]:
# primeros 17 digitos iguales, pero con floats: resultado erroneo
123456789012345675454325. - \
123456789012345678794305.

**Error catastrófico en la ecuación cuadrática**: este es un ejemplo clásico de la cancelación sustractiva, considere la ecuación cuadrática

$$ax^{2}+bx+c=0$$

la cual tiene solución exacta

$$x=\frac{-b \pm \sqrt{b^2-4ac} }{2a}.$$

El error de cancelación sustractiva aparece en una de las raíces cuando $b^2>>4ac$ debido a que en la raíz cuadrada sus términos aproximadamente se cancelan, (lo cual da $b \pm\sqrt{b^2}\approx 0$ dependiendo del caso). Una solución es reescribir la solución de la siguiente manera

$$x=\frac{-2c}{b \pm \sqrt{b^2-4ac} }.$$

Esta expresión tiene mayor precisión pues por ejemplo si $b>0$, entonces $b+\sqrt{b^2}\approx 2b$, igual en el otro caso $b<0$ pues $-b-\sqrt{b^2}\approx -2b$ (ver ejercicio al final). 

**Ejemplo**: Consideremos la siguiente ecuación,

$$x^2 - 1.786737601482363x + 2.054360090947453\times 10^{-8}=0$$

que tiene raíces analíticas (representación de double con 16 dígitos de precisión),

$$(x-1.786737589984535)(x-1.149782767465722\times 10^{-8})=0$$

Comparemos los dos métodos de solución:

In [ ]:
## solución de la ecuación ax^2 + bx + c = 0
# note que b**2 >> 4ac
x1=1.786737589984535    # raíz analítica
x2=1.149782767465722e-8 # 
a=1.; b = - 1.786737601482363; c = 2.054360090947453e-8 

print( b**2,'>>', 4*a*c)

K = b**2. - 4.*a*c
x_mas   = (-b + K**0.5)/(2*a)# note que b<0 y no hay cancelación 
x_menos = (-b - K**0.5)/(2*a)# raiz con problema de cancelación

y_mas   = -2.*c/(b + K**0.5) # no funciona, pues introduce cancelación (funcionaría si b>0).
y_menos = -2.*c/(b - K**0.5) # soluciona el problema de cancelación
print ("Formula normal:    ",x_menos, x_mas)
print ("Formula modificada:",y_menos, y_mas)
print ("valor real:        ",x2, x1)

Si analizamos con detalle, vemos que en el caso de `x_menos` solo se retienen 8 cifras significativas con la fórmula normal debido a la cancelación sustractiva entre $b$ y $\sqrt{(b^2-4ac)}$, pero al usar la fórmula modificada se recuperan todos los dígitos. Entonces, la solución es dada por los valores de `x_mas` de la fórmula original y `y_menos` de la fórmula modificada.

Note que la fórmula anterior solo evita la cancelación entre $b$ y $\sqrt{(b^2-4ac)}$ pero no la cancelación dentro de la raíz $b^2-4ac$, en estos casos se debe aumentar la precisión al doble (128 bits) si se requiere un resultado muy preciso. Veamos un ejemplo, considere la ecuación propuesta por Kajan,

$$94906265.625x^2-189812534x+94906268.375=0$$

con raíces

 $$(x - 1.000000028975958)(x- 1.000000000000000)=0$$ 

Si solucionamos en python vemos que las dos fórmulas dan resultados equivocados:

In [ ]:
a=94906265.625; b=-189812534; c = 94906268.375

K = b**2. - 4*a*c
x_mas   = (-b + K**0.5)/(2*a) 
x_menos = (-b - K**0.5)/(2*a)  

y_mas   = -2*c/(b + K**0.5) 
y_menos = -2*c/(b - K**0.5)  

print ("Formula normal:    ",x_mas, x_menos)
print ("Formula modificada:",y_mas, y_menos)

# Complemento
Definición de [ULP](https://en.wikipedia.org/wiki/Unit_in_the_last_place) ("*Unit in the Last Place*" o "*Unit of Least Precision*"): es una medida del espaciamiento entre dos números flotantes,  que es usada para medir la precisión en cálculos numéricos. 

<!---
In computer science and numerical analysis, unit in the last place or unit of least precision (ULP) is the spacing between floating-point numbers, i.e., the value the least significant digit (rightmost digit) represents if it is 1. It is used as a measure of accuracy in numeric calculations.
also see:
https://math.stackexchange.com/questions/42920/efficient-and-accurate-approximation-of-error-function
--->
En el siguiente ejemplo el resultado da $2^{53}$ debido al formato de doble precisión que usa 53 bits en el significando:

In [ ]:
x = 1.0 # valor inicial
n = 0   # exponente de 2^n
while x != x + 1: 
    x = x * 2 
    n = n + 1 

x, n 


## Ejercicios
1) Calcule el error absoluto y relativo de  $p$  y $p^∗$:

&emsp; a) $p = \pi,\, p^∗ = 22/7$<br> 
&emsp; b) $p = \pi,\, p^∗ = 3.1416$<br>
&emsp; c) $p = e,\, p∗ = 2.718$<br> 
&emsp; d) $p = \sqrt{2},\, p∗ = 1.414$<br>
&emsp; e) $p = e^{10},\, p^∗ = 22000$<br> 
&emsp; f) $p = 10^{\pi} ,\, p^∗ = 1400$<br>
&emsp; g) $p = 8!\,, p^∗ = 39900$<br> 
&emsp; h) $p = 9!,\, p^∗ = \sqrt{18\pi}(9/e)^9$

2) Considere el siguiente código:
```python      
for x in range(20):
    print (x,10**x + 1.0e20)
```
Al ejecutarlo los primeros 4 prints son iguales, explique porque.
¿Qué pasa si se usa 32 bits en la operación?

3) ¿Cuál es la cantidad más pequeña, $a$, que se puede agregar a la suma para que esta cambie (es decir $x+a\neq x$) los siguientes números? 
```python 
1.0e10
1.0e15
1.0e20
1.025e30
```        
si a) los números son de 64 bits b) son de 32 bits?

4) Explique por qué $ (1000 + 0.5) + 0.5 = 1000 + 0.5$ da una respuesta diferente a $1000. + (0.5 + 0.5)$ si solo se consideran 4 cifras significativas ¿Cuales son los resultados en ambos casos?

5) investigue como afecta la cancelación sustractiva la siguiente operación
$$\frac{f(b)-f(a)}{b-a},$$ si $f(x)=x^2, b=3.0$ y 

&emsp; a) $b-a=0.001$,<br> 
&emsp; b) $b-a=0.00001$,<br> 
&emsp; c) $b-a=0.000001$.  

6) Implemente las siguientes expreciones y series en python de manera tradicional, luego use la idea de [reciclaje](#reciclaje_de_variable) del paso anterior (como se hizo en la serie seno) para reducir el error y simplificar los cálculos (usar `eps = 1e-8`, $x$ = 45 y $N=100$), cálcular el error en cada caso,

&emsp; a) $e^x = \sum_{n=0}^N \frac{x^n}{n!}$. 

&emsp; b) $\binom {n}{k}={\frac {n!}{k!\,(n-k)!}}={\frac {n+1-k}{k}}{\binom {n}{k-1}}$, fórmula recurrente para coeficiente binomial (considere la simetría: $\binom {n}{k} =\binom {n}{n-k}$).

&emsp; c) $B_{k}=-\sum _{i=0}^{k-1}{k \choose {i}}{\frac {B_{i}}{k+1-i}}$, con $B_0=1$, fórmula recurrente para números de Bernouilli.

&emsp; d) $\cos x = \sum^{N}_{n=0} \frac{(-1)^n}{(2n)!} x^{2n}$.
     
&emsp; e) $\tan x = \sum^{N}_{n=1} \frac{B_{2n} (-4)^n \left(1-4^n\right)}{(2n)!} x^{2n-1}$, donde los $B_k$  son los números de Bernouilli. 
     
&emsp; f) $\text{arcsen}\, x = \sum^{N}_{n=0} \frac{(2n)!}{4^n (n!)^2 (2n+1)} x^{2n+1},\quad\mbox{ para } \left| x \right| < 1$. 
     
&emsp; g) $\arccos x =\frac{\pi}{2}-\text{arcsen}\, x =\frac{\pi}{2}- \sum^{N}_{n=0} \frac{(2n)!}{4^n (n!)^2 (2n+1)} x^{2n+1}$.

&emsp; h) $\arctan x = \sum^N_{n=0} \frac{(-1)^n}{2n+1} x^{2n+1}\quad\mbox{, para } \left| x \right| < 1.$

&emsp; i) $(x+y)^{n}=\sum _{k=0}^{n}{\binom {n}{k}}x^{n-k}y^{k}$, fórmula binomial.

&emsp; j) $\sum _{k=0}^{n}{\binom {n}{k}}=2^{n}$.

7) El triángulo de pascal se determina a partir de los coeficientes de la expansión de la fórmula binomial
dada en el ejercicio anterior, ósea:
$$ 
\begin{eqnarray}
(a+b)^{0}&=&\quad\quad\quad\quad\,\,\, 1\\
(a+b)^{1}&=&\quad\quad\quad\,\, 1a+1b\\
(a+b)^{2}&=&\quad\quad\, 1a^{2}+2ab+1b^{2}\\
(a+b)^{3}&=&\quad 1a^{3}+3a^{2}b+3ab^{2}+1b^{3}\\
&\vdots&\\
(a+b)^{n}&=&1a^{n}+a^{n-1}b+a^{n-2}b^{2}...+ 1b^{n},\\
\end{eqnarray}
$$

haga un programa que grafique el triángulo de pascal para $n=10$.

8) Escriba un programa que calcule las 2 raíces de la ecuación cuadática usando la solución numérica tradicional y otro con la solución que da mayor precisión, compare los resultados. a) Para esto use la ecuación $8.47x^2+52.31x+0.3904=0.0$.

b) Investigue como cambia el error a medida que la cancelación sustractiva se aproxima al epsilon de la máquina (use $a=1,b=1$, $c=10^{-n}$ con $n=1,2,3,...$).

9) La [suma de Kajan](https://en.wikipedia.org/wiki/Kahan_summation_algorithm) es un método numérico que reduce significativamente el error cuando se suman cantidades pequeñas con grandes en un arrego o lista de números tipo flotante. a) Implimente la suma de Kajan dada por el seudocódigo,
```c
function KahanSum(input)  // input is an array of dimension N.
    sum = 0.0             // Prepare the accumulator.
    c = 0.0               // A running compensation for lost low-order bits.
    for i = 1 to N do     // The array input has elements indexed input[1] to input[N].
        y = input[i] - c  // c is zero the first time around.
        t = sum + y       // sum is big, y small, so low-order digits of y are lost.
        c = (t - sum) - y // (t - sum) cancels the high-order part of y; subtracting y recovers negative (low part of y)
        sum = t           // Algebraically, c should always be zero. Beware overly-aggressive optimizing compilers!
    next i                // Next time around, the lost low part will be added to y in a fresh attempt.
    return sum
```  
b) Verifique que si `eps=1.1102230246251565e-16`, 
```c
(1.0 + eps) - eps da 0.9999999999999999
```
pero la suma de Kajam da 1.0 (encuentre el `eps` de su computador y pruebe).<br>
c) Muestre que si $a, b, c$ son $10000.0, 3.14159, 2.71828$, entonces $(a + b) + c$ da el valor $10005.8$ pero al suma de Kajam da el valor más preciso $10005.9$. (Nota, para ver esta diferencia necesitará trabajar con solo 6 cifras significativas en python3, para ello use,
```python
>>> from decimal import *
>>> getcontext().prec = 6
>>> a, b, c = [Decimal(n) for n in '10000.0 3.14159 2.71828'.split()]
```
10) Cuando en python se ejecuta el arreglo,
```python
np.array([1., 10**100, 1., -10**100]).sum() da 0.0
```
pero una simple impección del arreglo muestra que el resultado correcto es 2.0. Muestre que la suma de Kahan falla pero el algoritmo de *Neumaier* da el resultado correcto:

```c
function NeumaierSum(input)           // input is an array of dimension N.
    sum = 0.0
    c = 0.0                           // A running compensation for lost low-order bits.
    for i = 1 to N do
        t = sum + input[i]
        if |sum| >= |input[i]| then
            c += (sum - t) + input[i] // If sum is bigger, low-order digits of input[i] are lost.
        else
            c += (input[i] - t) + sum // Else low-order digits of sum are lost
        endif
        sum = t
    next i
    return sum + c                    // Correction only applied once in the very end
```

11) La fórmula de Arquímedes aproxima el número $\pi$ mediante el cálculo de perimetros inscritos en un círculo como,

$$
\pi \sim 6t_{i}2^{i} 
$$, 

con, $i=0,1, ...,n,$ donde, 

$$
t_{i+1}=\frac{{\sqrt {t_{i}^{2}+1}}-1}{t_{i}}.
$$

a) Muestre que $t_{i+1}$ se puede reescribir como,

$$
t_{i+1}=\frac{t_{i}}{{\sqrt {t_{i}^{2}+1}}+1}.
$$

b) Para $n=30$ y $t_{0}={\frac{1}{\sqrt{3}}}$ compare las dos fórmulas ¿cuál es más precisa? Grafique el error relativo y explique porque una es más precisa que la otra (use como valor téorico $\pi= 3.14159265358979323846$).


**Bibliografía**:

Libro Landau, Páez "*A Survey of
Computational Physics
Introductory Computational Science*", cap 1,2.

Libro Burden, *Numerical Analysis. cap 1.*

https://ece.uwaterloo.ca/~dwharder/NumericalAnalysis/02Numerics/Weaknesses/

https://en.wikipedia.org/wiki/Loss_of_significance

https://en.wikipedia.org/wiki/Floating-point_arithmetic#Floating-point_arithmetic_operations